In [56]:
import pandas as pd
from scipy import stats
import numpy as np

employees = pd.read_csv('employees.csv')
attrition = pd.read_csv('attrition_log.csv')
performance = pd.read_csv('performance.csv')
engagement = pd.read_csv('engagement.csv')

attrition["exit_date"] = pd.to_datetime(attrition["exit_date"])

In [30]:
employees["is_leaver"] = employees["status"].str.lower() == "departed"

# Dates
employees["hire_date"] = pd.to_datetime(employees["hire_date"])
attrition["exit_date"] = pd.to_datetime(attrition["exit_date"])

# Ordinal performance scale
RATING_ORDER = {
    "Unsatisfactory": 1,
    "Below Expectations": 2,
    "Meets Expectations": 3,
    "High Performer": 4,
    "Outstanding": 5,
}
performance["rating_score"] = performance["performance_rating"].map(RATING_ORDER)
assert performance["rating_score"].isna().sum() == 0, "Unmapped rating label found"

print("is_leaver value counts:")
print(employees["is_leaver"].value_counts())

is_leaver value counts:
is_leaver
False    12003
True      1400
Name: count, dtype: int64


In [31]:
print(employees["department"].unique())

<StringArray>
[   'Wealth Management',            'Insurance', 'Corporate Operations',
           'Technology',       'Retail Banking',    'Risk & Compliance',
 'Executive Leadership']
Length: 7, dtype: str


In [32]:
DEPT_NAME = "Risk & Compliance"
rc_employees = employees[employees["department"] == DEPT_NAME].copy()
print(f"{DEPT_NAME} headcount (all-time, incl. leavers): {len(rc_employees)}")
print(rc_employees["is_leaver"].value_counts())

Risk & Compliance headcount (all-time, incl. leavers): 2022
is_leaver
False    1783
True      239
Name: count, dtype: int64


In [33]:
print(attrition.columns.tolist())

print(attrition.index)
print(attrition.columns.tolist())

['employee_id', 'exit_date', 'exit_type', 'stated_exit_reason', 'notice_period_served', 'regrettable_flag', 'performance_band_at_exit', 'salary_at_exit', 'manager_id_at_exit', 'pathway']
RangeIndex(start=0, stop=1400, step=1)
['employee_id', 'exit_date', 'exit_type', 'stated_exit_reason', 'notice_period_served', 'regrettable_flag', 'performance_band_at_exit', 'salary_at_exit', 'manager_id_at_exit', 'pathway']


In [34]:
rc_leavers = rc_employees[rc_employees["is_leaver"]].merge(
    attrition[["employee_id", "exit_date", "exit_type", "regrettable_flag", "stated_exit_reason", "pathway"]],
    on="employee_id", how="left"
)

# print("exit_date" in employees.columns)
# print(rc_leavers.columns.tolist())

# compare = rc_leavers[["employee_id", "exit_date_x", "exit_date_y"]]
# print(compare.head(10))
# print("\nRows where they differ:", (compare["exit_date_x"] != compare["exit_date_y"]).sum())

# compare = rc_leavers[["employee_id", "exit_date_x", "exit_date_y"]]
# print(compare.head(10))
# print("\nRows where they differ:", (compare["exit_date_x"] != compare["exit_date_y"]).sum())

In [35]:
rc_leavers = rc_employees[rc_employees["is_leaver"]].drop(columns=["exit_date"], errors="ignore").merge(
    attrition[["employee_id", "exit_date", "exit_type", "regrettable_flag", "stated_exit_reason", "pathway"]],
    on="employee_id", how="left"
)

rc_leavers["exit_date"] = pd.to_datetime(rc_leavers["exit_date"])
rc_leavers["hire_date"] = pd.to_datetime(rc_leavers["hire_date"])

rc_leavers["working_days"] = (rc_leavers["exit_date"] - rc_leavers["hire_date"]).dt.days

print(f"R&C leavers: {len(rc_leavers)}")
print("\nWorking days summary:")
print(rc_leavers["working_days"].describe())

missing_exit = rc_leavers["exit_date"].isna().sum()
if missing_exit:
    print(f"\nWARNING: {missing_exit} R&C leaver(s) have no matching attrition record — working_days is NaN for these.")

R&C leavers: 239

Working days summary:
count      239.000000
mean      4185.640167
std       4457.856854
min         24.000000
25%        254.500000
50%       2526.000000
75%       7912.500000
max      13594.000000
Name: working_days, dtype: float64


In [36]:
print("Stated exit reasons (count):")
print(rc_leavers["stated_exit_reason"].value_counts())

print("\nStated exit reasons (%):")
print(rc_leavers["stated_exit_reason"].value_counts(normalize=True).mul(100).round(1))

print("\nExit type (voluntary/involuntary):")
print(rc_leavers["exit_type"].value_counts(normalize=True).mul(100).round(1))

print("\nRegrettable vs non-regrettable:")
print(rc_leavers["regrettable_flag"].value_counts(normalize=True).mul(100).round(1))

Stated exit reasons (count):
stated_exit_reason
Career advancement                   113
Better opportunity                    51
Involuntary - performance             33
Work-life balance                      8
Personal reasons                       7
Involuntary - conduct                  6
Involuntary - restructure              5
Study/career change                    5
Compensation                           4
Role uncertainty / unclear future      4
Relocation                             3
Name: count, dtype: int64

Stated exit reasons (%):
stated_exit_reason
Career advancement                   47.3
Better opportunity                   21.3
Involuntary - performance            13.8
Work-life balance                     3.3
Personal reasons                      2.9
Involuntary - conduct                 2.5
Involuntary - restructure             2.1
Study/career change                   2.1
Compensation                          1.7
Role uncertainty / unclear future     1.7
Relocation

In [37]:
performance["review_date"] = pd.to_datetime(performance["review_date"])

last_review = (
    performance.sort_values("review_date")
    .groupby("employee_id")
    .tail(1)[["employee_id", "performance_rating", "rating_score"]]
)

rc_leavers_perf = rc_leavers.merge(last_review, on="employee_id", how="left")

missing_perf = rc_leavers_perf["rating_score"].isna().sum()
if missing_perf:
    print(f"NOTE: {missing_perf} R&C leaver(s) have no performance review on record.")

print("\nLast rating distribution among R&C leavers (%):")
rating_order_cols = ["Unsatisfactory", "Below Expectations", "Meets Expectations", "High Performer", "Outstanding"]
dist = rc_leavers_perf["performance_rating"].value_counts(normalize=True).mul(100).round(1)
print(dist.reindex(rating_order_cols))


Last rating distribution among R&C leavers (%):
performance_rating
Unsatisfactory         3.8
Below Expectations     9.6
Meets Expectations    42.7
High Performer        29.7
Outstanding           14.2
Name: proportion, dtype: float64


In [38]:
# High performer or above = rating_score >= 4 ("High Performer" or "Outstanding")
HIGH_PERFORMER_THRESHOLD = 4

n_high_leavers = (rc_leavers_perf["rating_score"] >= HIGH_PERFORMER_THRESHOLD).sum()
n_total_leavers_rated = rc_leavers_perf["rating_score"].notna().sum()

print(f"R&C leavers who are High Performer or above: {n_high_leavers} of {n_total_leavers_rated} rated "
      f"({n_high_leavers / n_total_leavers_rated * 100:.1f}%)")

R&C leavers who are High Performer or above: 105 of 239 rated (43.9%)


In [39]:
rc_all_perf = rc_employees.merge(last_review, on="employee_id", how="left")
rc_all_perf["is_high_performer"] = rc_all_perf["rating_score"] >= HIGH_PERFORMER_THRESHOLD

print("High-performer rate, R&C leavers vs stayers (%):")
print(rc_all_perf.groupby("is_leaver")["is_high_performer"].mean().mul(100).round(1))

ct_high = pd.crosstab(rc_all_perf["is_leaver"], rc_all_perf["is_high_performer"])
print("\nCounts:")
print(ct_high)

if ct_high.shape == (2, 2):
    chi2_h, p_high, dof_h, exp_h = stats.chi2_contingency(ct_high)
    print(f"\nChi-square (is_leaver x is_high_performer): chi2={chi2_h:.2f}, p={p_high:.4f}")
    if p_high < 0.05:
        print("  -> Statistically significant: high-performer rate differs between R&C leavers and stayers.")
    else:
        print("  -> Not statistically significant at alpha=0.05.")
else:
    print("\nNot enough variation in one group to run chi-square (check cell counts above).")

High-performer rate, R&C leavers vs stayers (%):
is_leaver
False    38.6
True     43.9
Name: is_high_performer, dtype: float64

Counts:
is_high_performer  False  True 
is_leaver                      
False               1095    688
True                 134    105

Chi-square (is_leaver x is_high_performer): chi2=2.31, p=0.1287
  -> Not statistically significant at alpha=0.05.


In [40]:
LOW_PERFORMER_THRESHOLD = 2

rc_all_perf["is_low_performer"] = rc_all_perf["rating_score"] <= LOW_PERFORMER_THRESHOLD

print("Low-performer rate, R&C leavers vs stayers (%):")
print(rc_all_perf.groupby("is_leaver")["is_low_performer"].mean().mul(100).round(1))

ct_low = pd.crosstab(rc_all_perf["is_leaver"], rc_all_perf["is_low_performer"])
print("\nCounts:")
print(ct_low)

if ct_low.shape == (2, 2):
    chi2_l, p_low, dof_l, exp_l = stats.chi2_contingency(ct_low)
    print(f"\nChi-square (is_leaver x is_low_performer): chi2={chi2_l:.2f}, p={p_low:.4f}")
    if p_low < 0.05:
        print("  -> Statistically significant: low-performer rate differs between R&C leavers and stayers.")
    else:
        print("  -> Not statistically significant at alpha=0.05.")
else:
    print("\nNot enough variation in one group to run chi-square (check cell counts above).")

Low-performer rate, R&C leavers vs stayers (%):
is_leaver
False    15.4
True     13.4
Name: is_low_performer, dtype: float64

Counts:
is_low_performer  False  True 
is_leaver                     
False              1508    275
True                207     32

Chi-square (is_leaver x is_low_performer): chi2=0.53, p=0.4672
  -> Not statistically significant at alpha=0.05.


In [41]:
print("VOLUNTARY LEAVERS: COUNT AND PERFORMANCE RATING")

# Check the exact labels in exit_type first
print(rc_leavers["exit_type"].unique())


VOLUNTARY LEAVERS: COUNT AND PERFORMANCE RATING
<StringArray>
['voluntary', 'involuntary']
Length: 2, dtype: str


In [42]:
VOLUNTARY_LABEL = "voluntary"  

# Count voluntary vs involuntary among R&C leavers
print("Exit type counts (R&C leavers):")
print(rc_leavers["exit_type"].value_counts())
print("\nExit type (%):")
print(rc_leavers["exit_type"].value_counts(normalize=True).mul(100).round(1))

# Isolate voluntary leavers
rc_voluntary = rc_leavers[rc_leavers["exit_type"] == VOLUNTARY_LABEL].copy()
print(f"\nVoluntary leavers in R&C: {len(rc_voluntary)} of {len(rc_leavers)} total leavers "
      f"({len(rc_voluntary) / len(rc_leavers) * 100:.1f}%)")

Exit type counts (R&C leavers):
exit_type
voluntary      195
involuntary     44
Name: count, dtype: int64

Exit type (%):
exit_type
voluntary      81.6
involuntary    18.4
Name: proportion, dtype: float64

Voluntary leavers in R&C: 195 of 239 total leavers (81.6%)


In [43]:

performance["review_date"] = pd.to_datetime(performance["review_date"])
last_review = (
    performance.sort_values("review_date")
    .groupby("employee_id")
    .tail(1)[["employee_id", "performance_rating", "rating_score"]]
)

rc_voluntary = rc_voluntary.merge(last_review, on="employee_id", how="left")

missing_perf = rc_voluntary["rating_score"].isna().sum()
if missing_perf:
    print(f"\nNOTE: {missing_perf} voluntary R&C leaver(s) have no performance review on record.")

print("\nRating distribution among voluntary R&C leavers (%):")
rating_order_cols = ["Unsatisfactory", "Below Expectations", "Meets Expectations", "High Performer", "Outstanding"]
dist = rc_voluntary["performance_rating"].value_counts(normalize=True).mul(100).round(1)
print(dist.reindex(rating_order_cols))

print("\nAverage rating_score, voluntary R&C leavers:")
print(rc_voluntary["rating_score"].describe())


Rating distribution among voluntary R&C leavers (%):
performance_rating
Unsatisfactory         2.6
Below Expectations    10.8
Meets Expectations    44.6
High Performer        29.7
Outstanding           12.3
Name: proportion, dtype: float64

Average rating_score, voluntary R&C leavers:
count    195.000000
mean       3.384615
std        0.925453
min        1.000000
25%        3.000000
50%        3.000000
75%        4.000000
max        5.000000
Name: rating_score, dtype: float64


In [44]:
print("\nAverage rating_score: voluntary vs involuntary R&C leavers")
print(rc_leavers.merge(last_review, on="employee_id", how="left")
      .groupby("exit_type")["rating_score"].describe()[["count", "mean", "std", "50%"]])

# High performer share, voluntary vs involuntary
rc_leavers_perf = rc_leavers.merge(last_review, on="employee_id", how="left")
rc_leavers_perf["is_high_performer"] = rc_leavers_perf["rating_score"] >= 4

print("\nHigh-performer rate (%), voluntary vs involuntary R&C leavers:")
print(rc_leavers_perf.groupby("exit_type")["is_high_performer"].mean().mul(100).round(1))


Average rating_score: voluntary vs involuntary R&C leavers
             count      mean       std  50%
exit_type                                  
involuntary   44.0  3.522727  1.171138  4.0
voluntary    195.0  3.384615  0.925453  3.0

High-performer rate (%), voluntary vs involuntary R&C leavers:
exit_type
involuntary    52.3
voluntary      42.1
Name: is_high_performer, dtype: float64


In [45]:
summary = pd.DataFrame([
    {"metric": "R&C leavers - median working days", "value": rc_leavers["working_days"].median()},
    {"metric": "R&C leavers - % High Performer or above", "value": round(n_high_leavers / n_total_leavers_rated * 100, 1)},
    {"metric": "R&C - high-performer rate (stayers, %)", "value": round(rc_all_perf.loc[~rc_all_perf['is_leaver'], 'is_high_performer'].mean() * 100, 1)},
    {"metric": "R&C - high-performer rate (leavers, %)", "value": round(rc_all_perf.loc[rc_all_perf['is_leaver'], 'is_high_performer'].mean() * 100, 1)},
    {"metric": "R&C - low-performer rate (stayers, %)", "value": round(rc_all_perf.loc[~rc_all_perf['is_leaver'], 'is_low_performer'].mean() * 100, 1)},
    {"metric": "R&C - low-performer rate (leavers, %)", "value": round(rc_all_perf.loc[rc_all_perf['is_leaver'], 'is_low_performer'].mean() * 100, 1)},
])
print(summary.to_string(index=False))

                                 metric  value
      R&C leavers - median working days 2526.0
R&C leavers - % High Performer or above   43.9
 R&C - high-performer rate (stayers, %)   38.6
 R&C - high-performer rate (leavers, %)   43.9
  R&C - low-performer rate (stayers, %)   15.4
  R&C - low-performer rate (leavers, %)   13.4


In [46]:
print(engagement.columns.tolist())
print(engagement.head())

['employee_id', 'wave_number', 'survey_date', 'response_flag', 'manager_effectiveness', 'psychological_safety', 'recognition', 'career_development', 'senior_leadership_trust', 'purpose_meaning', 'wellbeing', 'confidence_in_role_future']
  employee_id  wave_number survey_date  response_flag  manager_effectiveness  \
0      E00001            1  2024-03-08           True                   5.00   
1      E00001            2  2024-06-27           True                   4.69   
2      E00001            3  2024-10-25           True                   4.65   
3      E00001            4  2025-01-31           True                   4.25   
4      E00001            5  2025-08-05           True                   4.57   

   psychological_safety  recognition  career_development  \
0                  4.39         4.55                4.25   
1                  4.26         4.10                3.52   
2                  4.67         4.87                3.72   
3                  5.00         5.00      

In [47]:
print("RESPONSE RATE BY DEPARTMENT AND WAVE")

dept_lookup = employees[["employee_id", "department"]].drop_duplicates()

resp_by_dept_wave = (
    engagement.merge(dept_lookup, on="employee_id", how="left")
    .groupby(["department", "wave_number"])["response_flag"]
    .mean() * 100
)
print(resp_by_dept_wave.unstack().round(1))

RESPONSE RATE BY DEPARTMENT AND WAVE
wave_number              1     2     3     4     5
department                                        
Corporate Operations  83.9  81.8  81.4  79.6  80.1
Executive Leadership  79.4  84.0  79.4  80.9  77.5
Insurance             82.9  81.4  79.7  81.8  82.4
Retail Banking        84.1  81.6  82.0  81.9  80.1
Risk & Compliance     84.4  84.1  80.3  80.4  80.0
Technology            83.5  82.4  81.0  81.1  80.1
Wealth Management     82.7  84.6  81.2  82.2  79.5


In [48]:
print("2. RESPONSE RATE TRAJECTORY: R&C VOLUNTARY LEAVERS vs STAYERS")

engagement["survey_date"] = pd.to_datetime(engagement["survey_date"])

rc_leaver_ids = rc_leavers.loc[rc_leavers["exit_type"] == "Voluntary", "employee_id"]
rc_stayer_ids = rc_employees.loc[~rc_employees["is_leaver"], "employee_id"]

# Leavers: attach exit_date, compute days_before_exit for each survey response
rc_leaver_eng = engagement[engagement["employee_id"].isin(rc_leaver_ids)].merge(
    rc_leavers[["employee_id", "exit_date"]], on="employee_id", how="left"
)
rc_leaver_eng["days_before_exit"] = (rc_leaver_eng["exit_date"] - rc_leaver_eng["survey_date"]).dt.days

# Only keep surveys that happened before the exit (exclude any stray post-exit rows)
rc_leaver_eng = rc_leaver_eng[rc_leaver_eng["days_before_exit"] >= 0]

# Bucket into "waves before exit" — waves look ~quarterly (Mar/Jun/Oct/Jan/Aug), so ~90-day bins
bins = [-1, 90, 180, 270, 360, 100_000]
labels = ["0-1 waves before exit", "1-2 waves before", "2-3 waves before", "3-4 waves before", "4+ waves before"]
rc_leaver_eng["exit_proximity"] = pd.cut(rc_leaver_eng["days_before_exit"], bins=bins, labels=labels)

leaver_trajectory = rc_leaver_eng.groupby("exit_proximity")["response_flag"].mean().mul(100).round(1)
print("\nVoluntary R&C leavers — response rate by proximity to exit (%):")
print(leaver_trajectory)

rc_stayer_resp = engagement[engagement["employee_id"].isin(rc_stayer_ids)]["response_flag"].mean() * 100
print(f"\nR&C stayers — overall response rate (%): {rc_stayer_resp:.1f}")

2. RESPONSE RATE TRAJECTORY: R&C VOLUNTARY LEAVERS vs STAYERS

Voluntary R&C leavers — response rate by proximity to exit (%):
Series([], Name: response_flag, dtype: float64)

R&C stayers — overall response rate (%): 82.5


In [49]:
print("RESPONSE RATE TRAJECTORY: R&C VOLUNTARY LEAVERS vs STAYERS")


rc_leaver_ids = rc_leavers.loc[rc_leavers["exit_type"] == "voluntary", "employee_id"]
rc_stayer_ids = rc_employees.loc[~rc_employees["is_leaver"], "employee_id"]

# For leavers: find their LAST wave_number (proxy for closest to exit), express other waves relative to it
rc_leaver_eng = engagement[engagement["employee_id"].isin(rc_leaver_ids)].copy()
last_wave_per_leaver = rc_leaver_eng.groupby("employee_id")["wave_number"].transform("max")
rc_leaver_eng["waves_before_exit"] = last_wave_per_leaver - rc_leaver_eng["wave_number"]

leaver_trajectory = (rc_leaver_eng.groupby("waves_before_exit")
                      .agg(response_rate=("response_flag","mean"), n=("response_flag","size")))
leaver_trajectory["response_rate"] *= 100
print("\nR&C voluntary leavers — response rate by waves before exit:")
print(leaver_trajectory)

# For stayers: same wave-relative framing, using their most recent wave as the anchor
rc_stayer_eng = engagement[engagement["employee_id"].isin(rc_stayer_ids)].copy()
last_wave_per_stayer = rc_stayer_eng.groupby("employee_id")["wave_number"].transform("max")
rc_stayer_eng["waves_before_last"] = last_wave_per_stayer - rc_stayer_eng["wave_number"]

stayer_trajectory = (rc_stayer_eng.groupby("waves_before_last")
                      .agg(response_rate=("response_flag","mean"), n=("response_flag","size")))
stayer_trajectory["response_rate"] *= 100
print("\nR&C stayers — response rate by waves before most recent wave:")
print(stayer_trajectory)

RESPONSE RATE TRAJECTORY: R&C VOLUNTARY LEAVERS vs STAYERS

R&C voluntary leavers — response rate by waves before exit:
                   response_rate    n
waves_before_exit                    
0                      67.721519  158
1                      72.807018  114
2                      70.786517   89
3                      78.333333   60
4                      83.333333   24

R&C stayers — response rate by waves before most recent wave:
                   response_rate     n
waves_before_last                     
0                      80.214205  1774
1                      81.601467  1636
2                      81.081081  1628
3                      85.357625  1482
4                      85.356880  1359


In [57]:
print("RESPONSE RATE TRAJECTORY: VOLUNTARY LEAVERS vs STAYERS, BY DEPARTMENT")

# sanity check exit_type values before filtering
print(leavers['exit_type'].unique())

def build_trajectory(emp_ids, wave_col_name):
    """Given a set of employee_ids, return response rate by waves-before-their-last-wave."""
    sub = engagement[engagement["employee_id"].isin(emp_ids)].copy()
    if sub.empty:
        return pd.DataFrame(columns=["response_rate", "n"])
    last_wave = sub.groupby("employee_id")["wave_number"].transform("max")
    sub[wave_col_name] = last_wave - sub["wave_number"]
    traj = sub.groupby(wave_col_name).agg(response_rate=("response_flag", "mean"),
                                           n=("response_flag", "size"))
    traj["response_rate"] *= 100
    return traj

departments = leavers['department'].dropna().unique()

results = {}  # store for summary table later

for dept in departments:
    dept_employees = leavers[leavers['department'] == dept].drop_duplicates('employee_id')

    dept_leaver_ids = dept_employees.loc[dept_employees['exit_type'] == 'voluntary', 'employee_id']
    dept_stayer_ids = dept_employees.loc[dept_employees['exit_type'].isna(), 'employee_id']

    leaver_traj = build_trajectory(dept_leaver_ids, "waves_before_exit")
    stayer_traj = build_trajectory(dept_stayer_ids, "waves_before_last")

    print(f"\n=== {dept} ===")
    print(f"Voluntary leavers (n employees={dept_leaver_ids.nunique()}):")
    print(leaver_traj)
    print(f"\nStayers (n employees={dept_stayer_ids.nunique()}):")
    print(stayer_traj)

    # capture wave-0 (closest to exit/most recent) vs wave-3 (furthest back, if available) for summary
    results[dept] = {
        'leaver_resp_at_0': leaver_traj['response_rate'].get(0, np.nan),
        'leaver_resp_at_3': leaver_traj['response_rate'].get(3, np.nan),
        'stayer_resp_at_0': stayer_traj['response_rate'].get(0, np.nan),
        'stayer_resp_at_3': stayer_traj['response_rate'].get(3, np.nan),
        'n_leavers': dept_leaver_ids.nunique(),
        'n_stayers': dept_stayer_ids.nunique(),
    }

# summary comparison across departments — sorted by steepest leaver decline
summary = pd.DataFrame(results).T
summary['leaver_decline_pts'] = summary['leaver_resp_at_3'] - summary['leaver_resp_at_0']
summary['stayer_decline_pts'] = summary['stayer_resp_at_3'] - summary['stayer_resp_at_0']
summary['leaver_vs_stayer_gap'] = summary['leaver_decline_pts'] - summary['stayer_decline_pts']

print("\n\n=== SUMMARY: response rate decline (percentage points) from 3 waves out to exit/last-wave ===")
display(summary.sort_values('leaver_vs_stayer_gap', ascending=False))

RESPONSE RATE TRAJECTORY: VOLUNTARY LEAVERS vs STAYERS, BY DEPARTMENT
<StringArray>
['voluntary']
Length: 1, dtype: str

=== Wealth Management ===
Voluntary leavers (n employees=105):
                   response_rate    n
waves_before_exit                    
0                      66.666667  105
1                      76.543210   81
2                      62.500000   56
3                      56.410256   39
4                      70.588235   17

Stayers (n employees=0):
Empty DataFrame
Columns: [response_rate, n]
Index: []

=== Corporate Operations ===
Voluntary leavers (n employees=115):
                   response_rate    n
waves_before_exit                    
0                      68.695652  115
1                      71.428571   84
2                      65.714286   70
3                      75.555556   45
4                      63.157895   19

Stayers (n employees=0):
Empty DataFrame
Columns: [response_rate, n]
Index: []

=== Insurance ===
Voluntary leavers (n employees=119):
 

,leaver_resp_at_0,leaver_resp_at_3,stayer_resp_at_0,stayer_resp_at_3,n_leavers,n_stayers,leaver_decline_pts,stayer_decline_pts,leaver_vs_stayer_gap
Wealth Management,66.666667,56.410256,NaN,NaN,105.0,0.0,-10.256410,NaN,NaN
Corporate Operations,68.695652,75.555556,NaN,NaN,115.0,0.0,6.859903,NaN,NaN
Insurance,71.428571,70.731707,NaN,NaN,119.0,0.0,-0.696864,NaN,NaN
Technology,59.624413,63.529412,NaN,NaN,213.0,0.0,3.904999,NaN,NaN
Risk & Compliance,67.721519,78.333333,NaN,NaN,158.0,0.0,10.611814,NaN,NaN
Retail Banking,64.545455,61.038961,NaN,NaN,220.0,0.0,-3.506494,NaN,NaN
Executive Leadership,80.000000,100.000000,NaN,NaN,15.0,0.0,20.000000,NaN,NaN
